# Enhanced SEPA Backtest with EDA Findings

This notebook incorporates findings from `scores_eda.ipynb` and `development_roadmap.md`:

| Finding | Implementation |
|---------|----------------|
| Effective threshold ~0.60 | `min_prob_elite` grid: [0.5, 0.6, 0.65] |
| M03 Trend pillar > 60 beats composite | Gate on `m03_pillar_trend` (not `m03_score`) |
| 5F exposure >= 0.75 is additive | Position scaling via `target_exposure` |
| 3+ consecutive days >= 0.60 | `score_persistence_days` entry rule |
| Score delta IC = 0.166 | `min_score_delta_5d` momentum filter |
| 1d return inverts | Skip T=0 breakout day entries |
| Exit when score < 0.435 | Score decay exit alongside SMA/stop |

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

## 1. Data Preparation

Load pre-computed caches with minimal columns to save memory.

In [ ]:
# Essential columns for backtest (skip 200+ feature columns)
SCORE_COLS = [
    'date', 'ticker', 'close',
    # Model outputs
    'prob_elite', 'calibrated_score', 'normalized_score', 'daily_pct_rank', 'trailing_pct',
    # Entry filters
    'breakout_ok', 'trend_ok',
    # M03 regime (pillar-level for gating)
    'm03_score', 'm03_pillar_trend', 'm03_pillar_liq', 'm03_pillar_risk',
    'm03_delta_5d', 'm03_delta_20d',
    # Forward returns (validation)
    'return_1d', 'return_5d', 'return_20d', 'return_60d',
    # Sector for analysis
    'sector', 'industry',
    # RS for ranking
    'rs_sector_rank', 'rs_industry_rank', 'rs_universe_rank',
]

# Load slim scores cache
scores_path = ROOT / 'notebooks' / 'scores_cache.parquet'
scores_df = pd.read_parquet(scores_path, columns=[c for c in SCORE_COLS if c != 'close'])
scores_df['date'] = pd.to_datetime(scores_df['date'])
print(f"Scores: {len(scores_df):,} rows, {scores_df['ticker'].nunique()} tickers")
print(f"Date range: {scores_df['date'].min().date()} to {scores_df['date'].max().date()}")
print(f"Memory: {scores_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
# Load 5-factor risk model
risk_path = ROOT / 'ideas' / 'risk_5_factor' / 'risk_scores.parquet'
risk_df = pd.read_parquet(risk_path)
risk_df = risk_df.reset_index().rename(columns={'index': 'date'})
risk_df['date'] = pd.to_datetime(risk_df['date'])

# Keep only relevant columns
RISK_COLS = ['date', 'veto_flag', 'target_exposure', 'rolling_percentile', 'weighted_z']
risk_df = risk_df[RISK_COLS].dropna()
print(f"Risk model: {len(risk_df):,} rows")
print(f"Date range: {risk_df['date'].min().date()} to {risk_df['date'].max().date()}")

In [ ]:
# Merge risk data onto scores (daily broadcast)
scores_df = scores_df.merge(risk_df, on='date', how='left')
print(f"Merged: {len(scores_df):,} rows")
print(f"Missing target_exposure: {scores_df['target_exposure'].isna().sum():,}")

In [ ]:
# Compute score momentum features (for entry timing)
scores_df = scores_df.sort_values(['ticker', 'date'])
scores_df['score_delta_5d'] = scores_df.groupby('ticker')['prob_elite'].diff(5)

# Compute consecutive days above threshold (for persistence rule)
def count_consecutive_above(group, threshold=0.60):
    above = (group['prob_elite'] >= threshold).astype(int)
    # Rolling count that resets on 0
    cumsum = above.cumsum()
    reset_points = cumsum.where(above == 0).ffill().fillna(0)
    return cumsum - reset_points

scores_df['consec_days_above_060'] = (
    scores_df.groupby('ticker', group_keys=False)
    .apply(count_consecutive_above)
    .astype(int)
)

print(f"Score momentum features added")
print(f"  score_delta_5d range: [{scores_df['score_delta_5d'].min():.3f}, {scores_df['score_delta_5d'].max():.3f}]")
print(f"  consec_days_above_060 max: {scores_df['consec_days_above_060'].max()}")

## 2. Enhanced Vectorized Backtest

Key enhancements:
- M03 pillar-level gating (not composite)
- 5-factor exposure scaling
- Score persistence entry rule
- Score decay exit

In [ ]:
from dataclasses import dataclass
from typing import Optional
import quantstats as qs

@dataclass
class BacktestConfig:
    """All backtest parameters in one place."""
    # Date range
    start_date: str = '2020-01-01'
    end_date: str = '2025-01-01'
    warmup_days: int = 10
    
    # Entry: score threshold
    min_prob_elite: float = 0.60
    
    # Entry: M03 regime gate (pillar-level)
    m03_gate: str = 'trend'  # 'none', 'trend', 'trend_liq', 'composite'
    m03_threshold: float = 60.0
    
    # Entry: 5-factor gate
    use_5f_gate: bool = True
    min_5f_exposure: float = 0.75
    
    # Entry: score persistence
    score_persistence_days: int = 0  # 0 = disabled, 3 = require 3+ consecutive days
    
    # Entry: score momentum
    min_score_delta_5d: float = 0.0  # 0 = disabled
    
    # Entry: skip breakout day (1d return inverts)
    skip_breakout_day: bool = False
    
    # Position sizing
    max_positions_per_day: int = 10
    initial_cash: float = 100_000
    position_size_pct: float = 0.10
    use_5f_sizing: bool = False  # Scale position by target_exposure
    
    # Exit: stop loss
    stop_loss_pct: float = 0.15
    
    # Exit: SMA trend
    sma_exit_period: int = 50
    
    # Exit: score decay
    use_score_exit: bool = False
    score_exit_threshold: float = 0.435  # Decile 9 floor
    
    # Exit: max hold
    max_hold_days: int = 252
    
    # Costs
    commission_pct: float = 0.001
    slippage_pct: float = 0.001

In [ ]:
import duckdb
import config

def run_enhanced_backtest(
    cfg: BacktestConfig,
    scores: pd.DataFrame,
    prices: Optional[pd.DataFrame] = None,
) -> tuple[pd.DataFrame, dict]:
    """
    Run enhanced vectorized backtest.
    
    Returns:
        trades: DataFrame with trade-level results
        stats: Summary statistics dict
    """
    # 1. Filter by date range
    df = scores[
        (scores['date'] >= cfg.start_date) & 
        (scores['date'] <= cfg.end_date)
    ].copy()
    
    # 2. Apply score threshold
    df = df[df['prob_elite'] >= cfg.min_prob_elite]
    
    # 3. Apply M03 regime gate
    if cfg.m03_gate == 'trend':
        df = df[df['m03_pillar_trend'] > cfg.m03_threshold]
    elif cfg.m03_gate == 'trend_liq':
        df = df[
            (df['m03_pillar_trend'] > cfg.m03_threshold) &
            (df['m03_pillar_liq'] > cfg.m03_threshold)
        ]
    elif cfg.m03_gate == 'composite':
        df = df[df['m03_score'] > cfg.m03_threshold]
    # 'none' = no filter
    
    # 4. Apply 5-factor gate
    if cfg.use_5f_gate:
        df = df[df['target_exposure'] >= cfg.min_5f_exposure]
    
    # 5. Apply score persistence
    if cfg.score_persistence_days > 0:
        df = df[df['consec_days_above_060'] >= cfg.score_persistence_days]
    
    # 6. Apply score momentum filter
    if cfg.min_score_delta_5d > 0:
        df = df[df['score_delta_5d'] >= cfg.min_score_delta_5d]
    
    # 7. Skip breakout day if configured
    if cfg.skip_breakout_day and 'breakout_ok' in df.columns:
        # Only keep rows where breakout_ok was True yesterday (T-1) not today
        df['breakout_yesterday'] = df.groupby('ticker')['breakout_ok'].shift(1)
        df = df[df['breakout_yesterday'] == True]
        df = df.drop(columns=['breakout_yesterday'])
    
    # 8. Apply warmup
    unique_dates = pd.Series(df['date'].unique()).sort_values().reset_index(drop=True)
    if len(unique_dates) <= cfg.warmup_days:
        return pd.DataFrame(), {'n_trades': 0, 'error': 'insufficient_dates'}
    warmup_cutoff = unique_dates.iloc[cfg.warmup_days]
    df = df[df['date'] >= warmup_cutoff]
    
    # 9. Rank and select top N per day
    df['daily_rank'] = df.groupby('date')['prob_elite'].rank(ascending=False, method='first')
    df = df[df['daily_rank'] <= cfg.max_positions_per_day]
    
    # 10. Deduplicate: first entry per ticker
    df = df.sort_values(['ticker', 'date'])
    entries = df.drop_duplicates(subset=['ticker'], keep='first')
    
    if entries.empty:
        return pd.DataFrame(), {'n_trades': 0, 'error': 'no_entries'}
    
    # Load prices if not provided
    if prices is None:
        db_path = config.DATA_DIR / 'market_data.duckdb'
        con = duckdb.connect(str(db_path), read_only=True)
        tickers = entries['ticker'].unique().tolist()
        prices = con.execute("""
            SELECT ticker, date, open, high, low, close
            FROM price_data
            WHERE ticker = ANY(?)
              AND date >= ? AND date <= ?
            ORDER BY ticker, date
        """, [tickers, cfg.start_date, cfg.end_date]).fetchdf()
        con.close()
        prices['date'] = pd.to_datetime(prices['date'])
    
    # Compute SMA for exit
    prices = prices.sort_values(['ticker', 'date'])
    prices['sma'] = (
        prices.groupby('ticker')['close']
        .transform(lambda s: s.rolling(cfg.sma_exit_period, min_periods=1).mean())
    )
    
    # Get entry prices
    entry_prices = prices.merge(
        entries[['ticker', 'date', 'prob_elite', 'calibrated_score', 'target_exposure']],
        on=['ticker', 'date'],
        how='inner',
    )[['ticker', 'date', 'close', 'prob_elite', 'calibrated_score', 'target_exposure']].rename(
        columns={'date': 'entry_date', 'close': 'entry_price', 'prob_elite': 'prob_elite_at_entry'}
    )
    
    if entry_prices.empty:
        return pd.DataFrame(), {'n_trades': 0, 'error': 'no_price_match'}
    
    # Simulate exits
    # Merge scores for score-based exit (if enabled)
    if cfg.use_score_exit:
        prices = prices.merge(
            scores[['date', 'ticker', 'prob_elite']].rename(columns={'prob_elite': 'current_score'}),
            on=['ticker', 'date'],
            how='left'
        )
    
    merged = prices.merge(
        entry_prices[['ticker', 'entry_date', 'entry_price', 'prob_elite_at_entry', 'target_exposure']],
        on='ticker',
        how='inner',
    )
    merged = merged[merged['date'] > merged['entry_date']].copy()
    
    # Exit conditions
    merged['stop_level'] = merged['entry_price'] * (1.0 - cfg.stop_loss_pct)
    merged['hit_stop'] = merged['low'] <= merged['stop_level']
    merged['hit_trend'] = merged['close'] < merged['sma']
    merged['bars_held'] = merged.groupby(['ticker', 'entry_date']).cumcount() + 1
    merged['hit_timeout'] = merged['bars_held'] >= cfg.max_hold_days
    
    if cfg.use_score_exit:
        merged['hit_score_decay'] = merged['current_score'] < cfg.score_exit_threshold
    else:
        merged['hit_score_decay'] = False
    
    # Priority: stop > score_decay > trend > timeout
    conditions = [
        merged['hit_stop'],
        merged['hit_score_decay'],
        merged['hit_trend'],
        merged['hit_timeout'],
    ]
    choices = ['stop_loss', 'score_decay', 'trend_break', 'max_hold']
    merged['exit_candidate'] = np.select(conditions, choices, default=None)
    
    exits = merged[merged['exit_candidate'].astype(str) != 'None'].copy()
    first_exits = exits.sort_values(['ticker', 'entry_date', 'date']).drop_duplicates(
        subset=['ticker', 'entry_date'], keep='first'
    )
    
    # Exit price
    first_exits['exit_price'] = np.where(
        first_exits['exit_candidate'] == 'stop_loss',
        first_exits['stop_level'],
        first_exits['close'],
    )
    
    # Build trades
    trades = entry_prices.merge(
        first_exits[['ticker', 'entry_date', 'date', 'exit_price', 'exit_candidate']],
        on=['ticker', 'entry_date'],
        how='left',
    ).rename(columns={'date': 'exit_date', 'exit_candidate': 'exit_reason'})
    
    # Mark open positions
    open_mask = trades['exit_date'].isna()
    if open_mask.any():
        last_prices = (
            prices.sort_values(['ticker', 'date'])
            .groupby('ticker')
            .tail(1)[['ticker', 'date', 'close']]
            .rename(columns={'date': 'last_date', 'close': 'last_close'})
        )
        trades = trades.merge(last_prices, on='ticker', how='left')
        trades.loc[open_mask, 'exit_date'] = trades.loc[open_mask, 'last_date']
        trades.loc[open_mask, 'exit_price'] = trades.loc[open_mask, 'last_close']
        trades.loc[open_mask, 'exit_reason'] = 'held_open'
        trades = trades.drop(columns=['last_date', 'last_close'], errors='ignore')
    
    # Compute PnL
    trades['pnl_pct'] = (trades['exit_price'] - trades['entry_price']) / trades['entry_price']
    trades['holding_days'] = (pd.to_datetime(trades['exit_date']) - pd.to_datetime(trades['entry_date'])).dt.days
    
    # Apply costs
    cost = 2 * (cfg.commission_pct + cfg.slippage_pct)
    trades['pnl_pct'] = trades['pnl_pct'] - cost
    
    # Position sizing with 5F scaling if enabled
    if cfg.use_5f_sizing:
        trades['position_size'] = cfg.position_size_pct * trades['target_exposure'].fillna(1.0)
    else:
        trades['position_size'] = cfg.position_size_pct
    
    # Compute stats
    wins = trades[trades['pnl_pct'] > 0]
    losses = trades[trades['pnl_pct'] <= 0]
    pf = abs(wins['pnl_pct'].sum() / losses['pnl_pct'].sum()) if len(losses) and losses['pnl_pct'].sum() != 0 else float('inf')
    
    # Simple equity curve
    returns_by_date = {}
    for _, t in trades.iterrows():
        exit_d = pd.Timestamp(t['exit_date'])
        pnl_dollar = cfg.initial_cash * t['position_size'] * t['pnl_pct']
        returns_by_date[exit_d] = returns_by_date.get(exit_d, 0.0) + pnl_dollar
    
    dates = pd.date_range(
        start=pd.Timestamp(trades['entry_date'].min()),
        end=pd.Timestamp(trades['exit_date'].max()),
        freq='B',
    )
    daily_pnl = pd.Series(0.0, index=dates)
    for d, v in returns_by_date.items():
        if d in daily_pnl.index:
            daily_pnl.loc[d] += v
        else:
            nearest = daily_pnl.index[daily_pnl.index >= d]
            if len(nearest):
                daily_pnl.loc[nearest[0]] += v
    equity = cfg.initial_cash + daily_pnl.cumsum()
    returns = equity.pct_change().dropna()
    
    total_ret = (equity.iloc[-1] - equity.iloc[0]) / equity.iloc[0] * 100 if len(equity) > 1 else 0
    sharpe = qs.stats.sharpe(returns) if len(returns) > 10 else 0
    max_dd = qs.stats.max_drawdown(returns) * 100 if len(returns) > 10 else 0
    
    stats = {
        'n_trades': len(trades),
        'n_tickers': trades['ticker'].nunique(),
        'win_rate': len(wins) / len(trades) if len(trades) > 0 else 0,
        'avg_pnl': trades['pnl_pct'].mean() if len(trades) > 0 else 0,
        'median_pnl': trades['pnl_pct'].median() if len(trades) > 0 else 0,
        'profit_factor': pf,
        'total_return': total_ret,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'avg_hold_days': trades['holding_days'].mean() if len(trades) > 0 else 0,
        'exit_reasons': trades['exit_reason'].value_counts().to_dict() if len(trades) > 0 else {},
    }
    
    return trades, stats

## 3. Baseline Run

In [ ]:
# Baseline: original parameters (for comparison)
cfg_baseline = BacktestConfig(
    start_date='2020-01-01',
    end_date='2025-01-01',
    min_prob_elite=0.50,
    m03_gate='none',
    use_5f_gate=False,
    stop_loss_pct=0.15,
    sma_exit_period=50,
    max_positions_per_day=10,
)

trades_baseline, stats_baseline = run_enhanced_backtest(cfg_baseline, scores_df)
print("BASELINE (no regime gating):")
for k, v in stats_baseline.items():
    if k == 'exit_reasons':
        print(f"  {k}: {v}")
    elif isinstance(v, float):
        print(f"  {k}: {v:.3f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# Enhanced: with EDA findings
cfg_enhanced = BacktestConfig(
    start_date='2020-01-01',
    end_date='2025-01-01',
    # Raised threshold per EDA
    min_prob_elite=0.60,
    # M03 Trend pillar gate (best single gate)
    m03_gate='trend',
    m03_threshold=60.0,
    # 5-factor additive gate
    use_5f_gate=True,
    min_5f_exposure=0.75,
    # Score persistence (best entry rule)
    score_persistence_days=3,
    # Stop & exit
    stop_loss_pct=0.15,
    sma_exit_period=100,
    max_positions_per_day=10,
)

trades_enhanced, stats_enhanced = run_enhanced_backtest(cfg_enhanced, scores_df)
print("\nENHANCED (with EDA findings):")
for k, v in stats_enhanced.items():
    if k == 'exit_reasons':
        print(f"  {k}: {v}")
    elif isinstance(v, float):
        print(f"  {k}: {v:.3f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# Compare
print(f"\n{'='*60}")
print("COMPARISON: Baseline vs Enhanced")
print(f"{'='*60}")
print(f"{'Metric':<20} {'Baseline':<15} {'Enhanced':<15} {'Delta':<15}")
print(f"{'-'*60}")
for key in ['n_trades', 'win_rate', 'avg_pnl', 'profit_factor', 'total_return', 'sharpe', 'max_dd']:
    b = stats_baseline.get(key, 0)
    e = stats_enhanced.get(key, 0)
    if isinstance(b, float):
        delta = e - b
        print(f"{key:<20} {b:<15.3f} {e:<15.3f} {delta:+.3f}")
    else:
        print(f"{key:<20} {b:<15} {e:<15}")

## 4. Parameter Sweep

Systematically test combinations of:
- Score threshold
- M03 gate type
- 5F gate
- Score persistence
- Stop loss
- SMA period

In [ ]:
import itertools
from tqdm.auto import tqdm

# Parameter grid
GRID = {
    'min_prob_elite': [0.50, 0.60, 0.65],
    'm03_gate': ['none', 'trend', 'trend_liq'],
    'use_5f_gate': [False, True],
    'score_persistence_days': [0, 3],
    'stop_loss_pct': [0.10, 0.15, 0.20],
    'sma_exit_period': [50, 100],
}

# Fixed params
FIXED = dict(
    start_date='2020-01-01',
    end_date='2025-01-01',
    m03_threshold=60.0,
    min_5f_exposure=0.75,
    max_positions_per_day=10,
    initial_cash=100_000,
    position_size_pct=0.10,
)

keys = list(GRID.keys())
combos = list(itertools.product(*GRID.values()))
print(f"Running {len(combos)} parameter combinations...")

In [ ]:
results = []

for combo in tqdm(combos, desc="Sweep"):
    params = dict(zip(keys, combo))
    cfg = BacktestConfig(**FIXED, **params)
    
    try:
        trades, stats = run_enhanced_backtest(cfg, scores_df)
        row = {
            **params,
            'n_trades': stats['n_trades'],
            'win_rate': stats['win_rate'],
            'avg_pnl': stats['avg_pnl'],
            'profit_factor': stats['profit_factor'],
            'total_return': stats['total_return'],
            'sharpe': stats['sharpe'],
            'max_dd': stats['max_dd'],
        }
    except Exception as e:
        row = {
            **params,
            'n_trades': 0,
            'win_rate': 0,
            'avg_pnl': 0,
            'profit_factor': 0,
            'total_return': 0,
            'sharpe': 0,
            'max_dd': 0,
            'error': str(e),
        }
    results.append(row)

results_df = pd.DataFrame(results)

In [ ]:
# Filter valid results and sort by Sharpe
valid = results_df[results_df['n_trades'] >= 20].copy()
valid = valid.sort_values('sharpe', ascending=False)

print(f"\n{'='*120}")
print(f"TOP 20 COMBINATIONS (by Sharpe, min 20 trades)")
print(f"{'='*120}")

display(valid.head(20).style.format({
    'win_rate': '{:.1%}',
    'avg_pnl': '{:+.2%}',
    'profit_factor': '{:.2f}',
    'total_return': '{:+.1f}%',
    'sharpe': '{:.2f}',
    'max_dd': '{:.1f}%',
}))

In [ ]:
# Analyze what matters most
print("\nAVERAGE SHARPE BY PARAMETER:")
print("="*40)

for param in keys:
    print(f"\n{param}:")
    agg = valid.groupby(param)['sharpe'].agg(['mean', 'std', 'count'])
    for val, row in agg.iterrows():
        print(f"  {val}: {row['mean']:.3f} +/- {row['std']:.3f} (n={int(row['count'])})")

## 5. Best Config Deep Dive

In [ ]:
# Extract best config from sweep
if len(valid) > 0:
    best_row = valid.iloc[0]
    best_params = {k: best_row[k] for k in keys}
    print("BEST CONFIG:")
    for k, v in best_params.items():
        print(f"  {k}: {v}")
    print(f"\n  Sharpe: {best_row['sharpe']:.2f}")
    print(f"  Total Return: {best_row['total_return']:.1f}%")
    print(f"  Win Rate: {best_row['win_rate']:.1%}")
    print(f"  Profit Factor: {best_row['profit_factor']:.2f}")

In [ ]:
# Run best config and show trades
if len(valid) > 0:
    cfg_best = BacktestConfig(**FIXED, **best_params)
    trades_best, stats_best = run_enhanced_backtest(cfg_best, scores_df)
    
    print(f"\nBEST CONFIG TRADES ({len(trades_best)} total):")
    print(trades_best.head(20).to_string())
    
    print(f"\nExit reasons:")
    print(trades_best['exit_reason'].value_counts())

## 6. Export Results

In [ ]:
# Save sweep results
results_df.to_parquet(ROOT / 'notebooks' / 'backtest_sweep_results.parquet', index=False)
print(f"Saved {len(results_df)} sweep results to backtest_sweep_results.parquet")

# Save best trades
if len(valid) > 0:
    trades_best.to_parquet(ROOT / 'notebooks' / 'best_config_trades.parquet', index=False)
    print(f"Saved {len(trades_best)} trades to best_config_trades.parquet")